# Bitcoin: explanations

Section 3.3, Table 5 and Fig. 1. All explanations refer to the Scenario 2 Random Forest of Table 3 and to its test partition.

* **SHAP**: path-dependent TreeSHAP; global importance is the mean absolute SHAP value for the fraud class. The direction column reports whether high or low values of a feature push towards fraud (sign of the correlation between the feature and its SHAP values).
* **CIU**: contextual importance and utility for the fraud output, averaged over 200 fraud wallets of the test partition.
* **DEXiRE**: rules extracted from an MLP distilled from the forest.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

from bfx.bitcoin import FEATURES
from bfx.modeling import SEED, ciu_means, distil_and_extract, readable_rule

DATA = ROOT / "data" / "bitcoin" / "scenarios"
RESULTS = ROOT / "results" / "bitcoin"
MODELS = ROOT / "models" / "bitcoin"

data = pd.read_csv(DATA / "scenario_2.csv")
train, test = train_test_split(data, test_size=0.2, stratify=data["y"], random_state=SEED)
X_train, X_test = train[FEATURES], test[FEATURES]
model = joblib.load(MODELS / "scenario_2_RF.joblib")
forest = model.named_steps["clf"]

## SHAP

In [ ]:
values = shap.TreeExplainer(forest, feature_perturbation="tree_path_dependent").shap_values(X_test)
values = values[:, :, 1] if values.ndim == 3 else values

direction = [np.corrcoef(X_test[f].rank(), values[:, i])[0, 1] for i, f in enumerate(FEATURES)]
ranking = pd.DataFrame({"feature": FEATURES, "mean_abs_shap": np.abs(values).mean(axis=0),
                        "fraud_signal": ["High ↑" if d > 0 else "Low ↓" for d in direction]})
ranking = ranking.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
ranking.index += 1
ranking.to_csv(RESULTS / "table5_shap.csv")
ranking.head(10).round(3)

In [ ]:
shap.summary_plot(values, X_test, feature_names=FEATURES, max_display=20, show=False)
plt.savefig(RESULTS / "fig1_shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

## CIU

In [ ]:
fraud_test = X_test[test["y"].to_numpy() == 1].sample(200, random_state=SEED)
ciu = ciu_means(model, X_train, fraud_test)
ciu.to_csv(RESULTS / "ciu_means.csv")
print(f"mean CI = {ciu['CI'].mean():.3f}   mean CU = {ciu['CU'].mean():.3f}")
ciu.head(10).round(3)

## DEXiRE

In [ ]:
rules, scaler, figures, rule_pred, covered = distil_and_extract(model, X_train.to_numpy(), X_test.to_numpy(), FEATURES)
rule_texts = [str(r) for r in rules.get_rules()]
figures["fraud_rules"] = sum(t.rstrip().endswith("1.0") or t.rstrip().endswith("1") for t in rule_texts)
figures["licit_rules"] = figures["n_rules"] - figures["fraud_rules"]
y_cov = test["y"].to_numpy()[covered]
figures["rule_accuracy"] = accuracy_score(y_cov, rule_pred[covered])
figures["rule_balanced_accuracy"] = balanced_accuracy_score(y_cov, rule_pred[covered])
_ = (RESULTS / "dexire.json").write_text(json.dumps(figures, indent=2, default=float))
_ = (RESULTS / "dexire_rules.txt").write_text("\n".join(readable_rule(t, scaler, FEATURES) for t in rule_texts))
pd.Series(figures).round(3)

The five shortest rules, with thresholds in original units:

In [ ]:
for t in sorted(rule_texts, key=len)[:5]:
    print(readable_rule(t, scaler, FEATURES))